# Create Delta tables, manipulating data and using constraints

In [ ]:
from pyspark.sql import Row

data = [Row(id=1, name="a"), Row(id=2, name="b")]
df = spark.createDataFrame(data)

# Managed Delta table
df.write.format("delta").mode("overwrite").saveAsTable("main.default.people")

In [ ]:
%sql
CREATE TABLE main.default.people_sql (id INT, name STRING) USING DELTA;
INSERT INTO main.default.people_sql VALUES (1, 'a'), (2, 'b');

In [ ]:
# Read
df = spark.read.table("main.default.people")
df.show()

# Write append
new_rows = spark.createDataFrame([(4, "d")], ["id", "name"])
new_rows.write.format("delta").mode("append").saveAsTable("main.default.people")

In [ ]:
%sql
# Read
SELECT * FROM main.default.people_sql;

# Write append
INSERT INTO main.default.people_sql VALUES (4, 'd');

In [ ]:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "main.default.people")
dt.update(condition="id = 2", set={"name": "b_updated"})

In [ ]:
%sql
UPDATE main.default.people_sql SET name = 'b_updated' WHERE id = 2;

In [ ]:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "main.default.people")
dt.delete("id = 1")

In [ ]:
%sql
DELETE FROM main.default.people_sql WHERE id = 1;

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Row

target = DeltaTable.forName(spark, "main.default.people")

updates_df = spark.createDataFrame([Row(id=2, name="b2"), Row(id=3, name="c")])

(target.alias("t")
 .merge(updates_df.alias("s"), "t.id = s.id")
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

updates_df.createOrReplaceTempView("updates_view")

In [ ]:
%sql
MERGE INTO main.default.people_sql AS t
USING updates_view AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [ ]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("event_date") \
  .saveAsTable("main.default.weblogs")

In [ ]:
# Create a weblogs table partitioned by event_date
CREATE TABLE main.default.weblogs (
  user_id INT,
  url STRING,
  status INT,
  event_time DATE,
  event_date DATE
)
USING DELTA PARTITIONED BY (event_date);

In [ ]:
%sql
# Dimension: customers (composite primary key)
CREATE TABLE customers (
  customer_id INT    NOT NULL,
  region_id   INT    NOT NULL,
  name        STRING NOT NULL,
  email       STRING,
  CONSTRAINT customers_pk PRIMARY KEY (customer_id, region_id)
);

# Fact: orders (single-column primary key, plus a composite FK to customers)
CREATE TABLE orders (
  order_id    INT      NOT NULL PRIMARY KEY,
  customer_id INT,
  region_id   INT,
  order_date  DATE     NOT NULL,
  amount      DECIMAL(12,2) NOT NULL,
  CONSTRAINT orders_customer_fk
    FOREIGN KEY (customer_id, region_id) REFERENCES customers
);

In [ ]:
%sql
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    first_name  STRING,
    last_name   STRING NOT NULL,
    age         INT,
    start_date  DATE,
    end_date    DATE,
    CONSTRAINT age_non_negative CHECK (age >= 0),
    CONSTRAINT valid_dates      CHECK (end_date > start_date)
);